# Lecture 02 Julia/Lux Orientation

This short notebook is the Julia entry point for the Lux-native translation
track. It uses the shared project in `julia/` and the same course conventions
as the Python notebooks:

```julia
RUN_MODE = "smoke"
SEED = 0
```

It is not a full Julia-language primer. It shows the conventions that later
`code_julia/` notebooks assume: Jupyter notebooks, the shared project environment,
feature-by-batch arrays, Lux's explicit parameter/state interface, and the
small shared training helper.

Every translated notebook activates the repository's shared Julia project rather
than carrying a local per-notebook environment. Keep this pattern when creating new
notebooks so all lectures use the same `DLEFJulia` helpers and dependency set.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, "..", "..", "..", "julia"))

using DLEFJulia
using Lux
using Optimisers
using StableRNGs

In [ ]:
RUN_MODE = "smoke"
SEED = 0

budget = run_mode_budget(RUN_MODE)
rng = rng_from_seed(SEED)

`RUN_MODE` controls the teaching budget. Use `smoke` for structural checks,
`teaching` for classroom-quality runs, and `production` only when reproducing a
full result. `SEED` should stay explicit so smoke checks and examples are
repeatable.

Lux layers consume arrays in feature-by-batch layout: rows are features and
columns are observations. Many Python notebooks use table-shaped
batch-by-feature arrays, so the translation converts at the boundary and keeps
the orientation explicit.

In [ ]:
batch_features = [
    -1.0  1.0
    -0.5  0.25
     0.0  0.0
     0.5  0.25
     1.0  1.0
]

x = to_feature_batch(batch_features)
target = reshape(batch_features[:, 1] .^ 2 .+ batch_features[:, 2], 1, :)

size(x), size(target)

In [ ]:
model = make_mlp(2, (8, 8), 1)
ps, st = setup_model(rng, model; parameter_type = Float64)

y, st_new = model(x, ps, st)
first_loss = mse_loss(y, target)

The shared training helper keeps Lux's explicit `model, ps, st` flow visible.
For full notebooks, lecture-specific residuals and diagnostics should stay
outside the optimizer loop.

In [ ]:
train_state = setup_training(model, ps, st, Optimisers.Adam(0.01))

loss_fn(model, ps, st) = begin
    prediction, st_next = model(x, ps, st)
    return mse_loss(prediction, target), st_next
end

history = NamedTuple[]
for _ in 1:budget.epochs
    metrics = train_step!(train_state, loss_fn; max_grad_norm = 10.0)
    append_metric!(history; step = metrics.step, loss = metrics.loss)
end

(initial_loss = first_loss, final_loss = history[end].loss, steps = length(history))

Next steps:

- For a fuller supervised-learning training loop, open
  `lecture_02_06_Lux_Training_Fundamentals.ipynb`.
- For the first economic residual notebook, open
  `../../lecture_03_deep_equilibrium_nets/code_julia/lecture_03_01_Brock_Mirman_1972_DEQN_Lux.ipynb`.
- When editing notebooks, keep them as Jupyter `.ipynb` files and avoid introducing
  Keras-, PyTorch-, or JAX-shaped wrapper APIs unless the notebook is explicitly
  comparing frameworks.